In [14]:
from assetextractor.extraction.utils import Config

from assetextractor.parsing.core.assets import Asset, AssetCache
from assetextractor.parsing.core.templates import Template
from assetextractor.parsing.core.attributes import ListAttribute

import csv
from pathlib import Path
import json
from lxml.etree import tostring

import typing as t

In [15]:
config = Config.from_json("config.json")
assets = AssetCache.load(config)
templates = assets.templates

In [16]:
params_path = Path("../anno-117-calculator/js/params.js")
with params_path.open(encoding="utf-8") as f:
    js_content = f.read()

prefix = 'if(window.params == null)window.params='
if not js_content.startswith(prefix):
    raise ValueError("params.js does not start with the expected prefix")
json_str = js_content[len(prefix):].strip()
if json_str.endswith(';'):
    json_str = json_str[:-1]

try:
    params = json.loads(json_str)
except Exception as e:
    raise ValueError(f"Failed to parse params.js as JSON: {e}")

product_to_factory = {}
product_region_to_factory = {}  # (product, region literal) -> first factory producing it there
product_to_inputs = {}
factory_to_upkeep = {}
factory_cycle_time = {}
product_to_price = {}

for factory in params.get("factories", []):
    try:
        guid = factory["guid"]
        if factory.get("outputs") is not None:
            product = factory.get("outputs")[0]["product"]
            # params.js no longer exports products[].mainFactory, fall back to the first factory producing the product
            product_to_factory.setdefault(product, guid)
            for region in factory.get("associatedRegions") or []:
                product_region_to_factory.setdefault((product, region), guid)
            product_to_inputs[product] = [(entry["product"], entry["amount"]) for entry in factory.get("inputs", [])]
            factory_to_upkeep[guid] = factory["maintenances"][0]["amount"]
            factory_cycle_time[guid] = factory["cycleTime"]
    except (KeyError, ValueError, TypeError):
        raise ValueError(f"Invalid factory entry: {guid}")

for prod in params.get("products", []):
    try:
        guid = int(prod["guid"])
        product_to_price[guid] = assets[guid].Product.BasePrice()
        if prod.get("mainFactory") is not None:
            main_factory = int(prod["mainFactory"])
            product_to_factory[guid] = main_factory
    except (KeyError, ValueError, TypeError):
        raise ValueError(f"Invalid product entry: {prod}")
    


In [17]:
def get_production_price(guid):
    factory = product_to_factory.get(guid)

    if factory is None:
        return None

    input_prices = [(get_production_price(input[0]), input[1]) for input in product_to_inputs[guid]]
    if any(price is None for price, amount in input_prices):
        return None

    return factory_to_upkeep[factory] / 60 * factory_cycle_time[factory]+ sum(price * amount for price, amount in input_prices)

In [18]:
factory = product_to_factory.get(2999)

In [19]:
len(product_to_inputs)

142

In [20]:
product_to_inputs[2180]

[(3158, 1)]

In [21]:
get_production_price(2145)

95.0

In [22]:
attributes = assets.datasets["NeedAttributeType"].literals

In [23]:
def english(text):
    if text is None:
        return None
    return text.values.get("english") if hasattr(text, "values") else str(text)

def format_area_effect(effect: Asset) -> str:
    """'<effect name>: <buff> <value>, ...' using Asset.buff_ui on every buff of the effect"""
    parts = []
    for buff_entry in effect.Effect.Buffs:
        buff = buff_entry.GUID()
        if buff is None:
            continue
        parts += [f"{english(ui.text)} {ui.value}".strip() for ui in buff.buff_ui]
    return f"{english(effect.text) or str(effect)}: {', '.join(parts)}"

# need_guid -> region literals (Roman / Celtic / Egyptian), taken from the residences listing the need
need_region_literals: dict[int, list[str]] = {}
for res_asset in templates["ResidenceBuilding"].assets:
    for entry in res_asset.Residence7.NeedsList:
        need_region_literals.setdefault(entry.Need.guid, res_asset.find_value("Building.AssociatedRegions") or [])

ui_text_cache = assets.properties.ui_text_cache
def region_name(literal: str) -> str:
    """session name (Latium / Albion / Aegyptus) of a region literal"""
    mapping = ui_text_cache.get_ui_text("Region", literal)
    return english(mapping.text) if mapping is not None and mapping.text is not None else literal

def need_factory(need_guid: int, product_guid: int):
    for literal in need_region_literals.get(need_guid, []):
        if (product_guid, literal) in product_region_to_factory:
            return product_region_to_factory[(product_guid, literal)]
    return product_to_factory.get(product_guid)

rows = [["region", "name"] + attributes + [f"Area Buff {attribute}" for attribute in attributes] + ["Area Effect"]]
empty_attributes = [None for i in range(len(attributes))]

for need in templates["Need"].assets:
    product = need.Need.NeedProduct()
    row = [", ".join(region_name(r) for r in need_region_literals.get(need.guid, [])) or None, str(product)] #region, name
    for attribute in attributes:
        val = need.find(f"Need.NeedAttributes.{attribute}.Value").value
        row.append(None if val is None or val == 0 else val)

    factory_guid = need_factory(need.guid, product.guid)
    if factory_guid is not None:
        attribute_values = {}
        for attribute in attributes:
            attribute_values[attribute] = 0

        area_effects = []
        for entry in assets.get(factory_guid).Building.FunctionalEffects:
            buff = entry.FunctionalEffect()
            area_effects.append(format_area_effect(buff))

            area_attributes = buff.Effect.Buffs[0].GUID().BuildingUpgrade.AdditionalAttributes
            for attribute in attributes:
                val = area_attributes[attribute].AmountOrPercent.value
                if val is not None:
                    attribute_values[attribute] += val

        row += [None if val == 0 else val for attr, val in attribute_values.items()]
        row.append("; ".join(area_effects) or None)
    else:
        row += empty_attributes + [None]

    rows.append(row)

In [24]:


output_path = Path("results/tables/needs.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", newline='', encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerows(rows)

In [25]:
output_path = Path("results/tables/products.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", newline='', encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["guid", "product", "production price", "base price"])
    for product in product_to_price:
        writer.writerow([product, str(assets[product]), get_production_price(product), product_to_price[product]])

In [26]:
# need_guid -> product_asset (skip abstract/service needs)
need_product: dict[int, object] = {}
for need_asset in templates["Need"].assets:
    try:
        product = need_asset.Need.NeedProduct()
        if product is None or product.Product.IsAbstract():
            continue
        need_product[need_asset.guid] = product
    except Exception:
        pass

# pop_level_guid -> english name
pop_level_name: dict[int, str] = {}
for pl_asset in templates["PopulationLevel"].assets:
    english = pl_asset.text.values.get("english") if pl_asset.text else None
    pop_level_name[pl_asset.guid] = english or pl_asset.find("Standard.Name")()

# one residence per pop level, sorted by guid for stable output
# NeedConsumptionRate is already in t/min per house at base difficulty (consumptionFactor=1)
seen_levels: set[int] = set()
rows = [["pop tier", "good", "t/min per house"]]

for res_asset in sorted(templates["ResidenceBuilding"].assets, key=lambda a: a.guid):
    try:
        pop_level_guid = res_asset.Residence7.PopulationLevel.guid
    except Exception:
        continue
    if pop_level_guid in seen_levels:
        continue
    seen_levels.add(pop_level_guid)
    tier_name = pop_level_name.get(pop_level_guid, str(pop_level_guid))

    for need in res_asset.Residence7.NeedsList:
        try:
            need_guid = need.Need.guid
        except Exception:
            continue
        if need_guid not in need_product:
            continue
        rate = need.NeedConsumptionRate()
        if not rate:
            continue
        product = need_product[need_guid]
        good_name = (product.text.values.get("english") if product.text else None) or str(product)
        rows.append([tier_name, good_name, round(rate, 6)])

output_path = Path("results/tables/consumption.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerows(rows)